In [2]:
"""
Cell 1: Setup and Loading Data
We will load the Parquet file we just created.
"""
# Cell 1
import pandas as pd
import numpy as np
import datetime as dt
import warnings
import os

warnings.filterwarnings('ignore')

# Define paths
PROCESSED_DATA_PATH = '../data/processed/cleaned_online_retail.parquet'
FEATURES_SAVE_PATH = '../data/processed/rfm_and_churn_labels.parquet'

print("Loading cleaned dataset...")
df = pd.read_parquet(PROCESSED_DATA_PATH)

print(f"Dataset Shape: {df.shape}")
print(f"Date range: from {df['Date'].min().date()} to {df['Date'].max().date()}")

Loading cleaned dataset...
Dataset Shape: (397885, 10)
Date range: from 2010-12-01 to 2011-12-09


In [3]:
"""
Cell 2: Defining the Time Windows
We will dynamically find the last date in the dataset and subtract 90 days to create our cutoff date.
"""
# Cell 2
# Find the absolute last date in the dataset
max_date = df['Date'].max()

# Define the performance window length (90 days is standard for e-commerce churn)
performance_days = 90
cutoff_date = max_date - pd.Timedelta(days=performance_days)

print(f"Dataset End Date: {max_date.date()}")
print(f"Cutoff Date (Splitting point): {cutoff_date.date()}")
print(f"Performance Window: {performance_days} days")

# Split the data
observation_data = df[df['Date'] < cutoff_date].copy()
performance_data = df[df['Date'] >= cutoff_date].copy()

print(f"\nTransactions in Observation Window: {len(observation_data)}")
print(f"Transactions in Performance Window: {len(performance_data)}")


Dataset End Date: 2011-12-09
Cutoff Date (Splitting point): 2011-09-10
Performance Window: 90 days

Transactions in Observation Window: 236358
Transactions in Performance Window: 161527


In [4]:
"""
Cell 3: Creating the Target Variable (Churn Labels)
Now we look at the customers in the observation window and check if they showed up in the performance window.
"""
# Cell 3
# 1. Get the list of all unique customers in the observation window
base_customers = pd.DataFrame(observation_data['CustomerID'].unique(), columns=['CustomerID'])

# 2. Get the list of customers who made a purchase in the performance window (The Retained ones)
retained_customers = performance_data['CustomerID'].unique()

# 3. Create the Churn Label
# If a customer is in the base list but NOT in the retained list, they churned (1). Otherwise, 0.
base_customers['Churn'] = np.where(base_customers['CustomerID'].isin(retained_customers), 0, 1)

churn_rate = base_customers['Churn'].mean() * 100
print(f"Total Base Customers: {len(base_customers)}")
print(f"Churned Customers: {base_customers['Churn'].sum()}")
print(f"Baseline Churn Rate: {churn_rate:.2f}%")

display(base_customers.head())

Total Base Customers: 3370
Churned Customers: 1449
Baseline Churn Rate: 43.00%


,CustomerID,Churn
0,17850,1
1,13047,0
2,12583,0
3,13748,1
4,15100,1


In [5]:
"""
Cell 4: Calculating RFM Features
We will calculate Recency, Frequency, and Monetary value strictly using the observation_data so there is zero data leakage.
"""

# Cell 4
print("Calculating RFM features...")

# For Recency, we calculate days relative to the cutoff_date, NOT today's date
rfm = observation_data.groupby('CustomerID').agg({
    'Date': lambda x: (cutoff_date - x.max()).days,  # Recency: Days since last purchase to cutoff
    'InvoiceNo': 'nunique',                          # Frequency: Count of unique invoices
    'TotalAmount': 'sum'                             # Monetary: Total spend
}).reset_index()

# Rename columns for clarity
rfm.rename(columns={
    'Date': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

display(rfm.head())

Calculating RFM features...


,CustomerID,Recency,Frequency,Monetary
0,12346,235,1,77183.60
1,12347,39,5,2790.86
2,12348,158,3,1487.24
3,12350,220,1,334.40
4,12352,172,5,1561.81


In [6]:
"""
Cell 5: Merging Features with Labels and Validating
We combine our RFM features with the Churn labels we created in Cell 3. We also add assert statements to guarantee we haven't lost any customers or created impossible values.
"""
# Cell 5
# Merge the features and the target label
df_modeling = pd.merge(rfm, base_customers, on='CustomerID', how='inner')

# --- PRODUCTION PIPELINE CHECKS ---
assert len(df_modeling) == len(base_customers), "Pipeline Error: Customer count mismatch after merge!"
assert df_modeling['Recency'].min() >= 0, "Pipeline Error: Negative Recency detected (Future leakage)!"
assert df_modeling['Frequency'].min() > 0, "Pipeline Error: Zero frequency detected!"
assert df_modeling['Churn'].isin([0, 1]).all(), "Pipeline Error: Churn label contains values other than 0 and 1!"

print("All pipeline assertions passed. Ready for modeling.")
display(df_modeling.head())

All pipeline assertions passed. Ready for modeling.


,CustomerID,Recency,Frequency,Monetary,Churn
0,12346,235,1,77183.60,1
1,12347,39,5,2790.86,0
2,12348,158,3,1487.24,0
3,12350,220,1,334.40,1
4,12352,172,5,1561.81,0


In [ ]:
"""
Cell 6: Saving the Modeling Dataset
Finally, we save this to a new Parquet file. This file now contains everything an ML model needs: the features (X) and the answers (y).
"""

# Cell 6
# Save to processed folder
df_modeling.to_parquet(FEATURES_SAVE_PATH, index=False)
print(f"Modeling dataset successfully saved to {FEATURES_SAVE_PATH}")

Modeling dataset successfully saved to ../data/processed/rfm_and_churn_labels.parquet


: 